# Notebook 5: Feature Engineering

This notebook builds the final features using only information available at prediction time. All transformations (imputers, scaler, encoder) are fit on the training split only, then applied to validation and test without refitting.

**Reads:** `train.csv`, `val.csv`, `test.csv`  
**Artifacts produced:** `features_train.csv`, `features_val.csv`, `features_test.csv`, `num_imputer.joblib`, `scaler.joblib`, `cat_imputer.joblib`, `encoder.joblib`, `feature_list.txt`

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
train = pd.read_csv("train.csv", parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
val = pd.read_csv("val.csv", parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
test = pd.read_csv("test.csv", parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])

In [3]:
# Only use information available at prediction time (order placed).
# Do not use order_delivered_customer_date or review score as a feature: they only exist
# after delivery and would leak the label.

In [4]:
def add_time_features(df):
    df = df.copy()
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    return df

In [5]:
train = add_time_features(train)
val = add_time_features(val)
test = add_time_features(test)

In [6]:
numeric_features = ["total_price", "total_freight", "n_items", "total_payment_value",
                     "n_payment_installments", "purchase_weekday", "purchase_month", "purchase_hour"]
categorical_features = ["customer_state", "payment_type", "product_category_name_english"]

In [7]:
# Impute numeric missing values using the training median
num_imputer = SimpleImputer(strategy="median")
train_num = num_imputer.fit_transform(train[numeric_features])
val_num = num_imputer.transform(val[numeric_features])
test_num = num_imputer.transform(test[numeric_features])

In [8]:
# Scale numeric features using training statistics
scaler = StandardScaler()
train_num = scaler.fit_transform(train_num)
val_num = scaler.transform(val_num)
test_num = scaler.transform(test_num)

In [9]:
# Impute categorical missing values with a constant, then one-hot encode
cat_imputer = SimpleImputer(strategy="constant", fill_value="unknown")
train_cat = cat_imputer.fit_transform(train[categorical_features])
val_cat = cat_imputer.transform(val[categorical_features])
test_cat = cat_imputer.transform(test[categorical_features])

In [10]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cat_enc = encoder.fit_transform(train_cat)
val_cat_enc = encoder.transform(val_cat)
test_cat_enc = encoder.transform(test_cat)

In [11]:
feature_names = numeric_features + list(encoder.get_feature_names_out(categorical_features))

In [12]:
X_train = np.hstack([train_num, train_cat_enc])
X_val = np.hstack([val_num, val_cat_enc])
X_test = np.hstack([test_num, test_cat_enc])

In [13]:
y_train = train["is_late"].values
y_val = val["is_late"].values
y_test = test["is_late"].values

In [14]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Number of features:", len(feature_names))

X_train shape: (67526, 111)
X_val shape: (14473, 111)
X_test shape: (14471, 111)
Number of features: 111


In [15]:
# Save the final feature tables
pd.DataFrame(X_train, columns=feature_names).assign(is_late=y_train).to_csv("features_train.csv", index=False)
pd.DataFrame(X_val, columns=feature_names).assign(is_late=y_val).to_csv("features_val.csv", index=False)
pd.DataFrame(X_test, columns=feature_names).assign(is_late=y_test).to_csv("features_test.csv", index=False)

In [16]:
# Save the fitted transformers, not just the output table.
# In production, the pipeline must load these same objects and never re-fit on new data.
joblib.dump(num_imputer, "num_imputer.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(cat_imputer, "cat_imputer.joblib")
joblib.dump(encoder, "encoder.joblib")

['encoder.joblib']

In [17]:
with open("feature_list.txt", "w") as f:
    f.write("\n".join(feature_names))

In [18]:
print("\nArtifacts saved: features_train.csv, features_val.csv, features_test.csv, "
      "num_imputer.joblib, scaler.joblib, cat_imputer.joblib, encoder.joblib, feature_list.txt")


Artifacts saved: features_train.csv, features_val.csv, features_test.csv, num_imputer.joblib, scaler.joblib, cat_imputer.joblib, encoder.joblib, feature_list.txt
